# Xenium Human Pancreas Data Exploration

This notebook provides an overview of the Xenium spatial transcriptomics dataset.
We visualize the morphology image, examine transcript distributions, and explore
the gene expression matrix.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from cpsam_xenium_analysis import config, data_loader as dl

%matplotlib inline
plt.rcParams.update({'figure.dpi': 150, 'font.size': 8})


## 1. Load Experiment Metadata


In [ ]:
metadata = dl.load_experiment_metadata()
print(f'Region: {metadata.get("region_name", "N/A")}')
print(f'Panel: {metadata.get("panel_name", "N/A")}')
print(f'Cells detected: {metadata.get("num_cells", "N/A"):,}')
print(f'Pixel size: {metadata.get("pixel_size", "N/A")} um/px')


## 2. Load Morphology Image (Cropped for Speed)


In [ ]:
crop = config.CROP_ROI  # 4000x4000 region for demo
morph_img = dl.load_morphology_image(use_focus=True, crop_roi=crop)
print(f'Morphology image shape: {morph_img.shape}, dtype={morph_img.dtype}')


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.imshow(morph_img)
ax.set_title('Morphology Image (Focus-Merged, Cropped ROI)')
ax.axis('off')
plt.show()


## 3. Load H&E Image


In [ ]:
he_img = dl.load_he_image(crop_roi=crop)
print(f'H&E image shape: {he_img.shape}, dtype={he_img.dtype}')


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.imshow(he_img)
ax.set_title('H&E Image (Cropped ROI)')
ax.axis('off')
plt.show()


## 4. Transcript Statistics


In [ ]:
transcripts = dl.load_transcripts(min_qv=20)
print(f'Total transcripts: {len(transcripts):,}')
print(f'Unique genes: {transcripts["feature_name"].nunique()}')
print(f'Assigned to cells: {(transcripts["cell_id"]!="UNASSIGNED").sum():,} ({100*(transcripts["cell_id"]!="UNASSIGNED").mean():.1f}%)')

transcripts.head()


In [ ]:
# Top 20 expressed genes
top_genes = transcripts['feature_name'].value_counts().head(20)
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
top_genes.plot(kind='barh', ax=ax)
ax.set_xlabel('Transcript count')
ax.set_title('Top 20 Expressed Genes')
plt.tight_layout()
plt.show()


## 5. Xenium Cell Summary


In [ ]:
cells = dl.load_cells_df()
print(f'Total Xenium cells: {len(cells):,}')
print()
print('Metrics summary:')
print(cells[['cell_area', 'nucleus_area', 'transcript_counts', 'total_counts']].describe())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(cells['cell_area'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Cell area (um^2)')
axes[0].set_ylabel('Count')
axes[0].set_title('Cell Area Distribution')

axes[1].hist(cells['transcript_counts'], bins=50, color='coral', edgecolor='white')
axes[1].set_xlabel('Transcript counts')
axes[1].set_title('Transcripts per Cell')

axes[2].scatter(cells['cell_area'], cells['transcript_counts'], s=1, alpha=0.3)
axes[2].set_xlabel('Cell area (um^2)')
axes[2].set_ylabel('Transcript counts')
axes[2].set_title('Area vs Transcripts')

plt.tight_layout()
plt.show()


## 6. Gene Expression Matrix


In [ ]:
expr_matrix, cell_ids, gene_names = dl.load_expression_matrix()
print(f'Expression matrix shape: {expr_matrix.shape}')
print(f'  Cells: {expr_matrix.shape[0]:,}')
print(f'  Genes: {expr_matrix.shape[1]}')
print(f'  Non-zero entries: {expr_matrix.nnz:,}')
print(f'  Sparsity: {100 * expr_matrix.nnz / (expr_matrix.shape[0] * expr_matrix.shape[1]):.2f}%')
print()
print(f'First 10 genes: {gene_names[:10]}')


In [ ]:
# Gene detection rate
detection_rate = np.array((expr_matrix > 0).sum(axis=0)).flatten() / expr_matrix.shape[0]
top_detected = pd.DataFrame({'gene': gene_names, 'detection_rate': detection_rate}).sort_values('detection_rate', ascending=False).head(15)

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.barh(range(len(top_detected)), top_detected['detection_rate'].values, color='teal')
ax.set_yticks(range(len(top_detected)))
ax.set_yticklabels(top_detected['gene'].values)
ax.set_xlabel('Fraction of cells expressing gene')
ax.set_title('Top 15 Most Widely Expressed Genes')
ax.invert_yaxis()
plt.tight_layout()
plt.show()
